In [1]:
import gym 
import bauwerk
import numpy as np

from agent.sac import SACAgent
from replay_buffer import ReplayBuffer
from cfgs.parser import parse_cfg
import utils
from utils import ObsWrapper
from tqdm import tqdm


/Users/scottjeen/miniforge3/envs/odes/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
env = gym.make("bauwerk/SolarBatteryHouse-v0")
obs = env.reset()

In [3]:
eval_length = 24 * 30
seed = 1

env = gym.make("bauwerk/BuildDistB-v0")
obs = env.reset()
print(env.cfg.battery_size)
# mean random performance over 100 trials
random_trials = [evaluate_actions([env.action_space.sample() for _ in range(eval_length)], seed) for _ in range(100)]
random_std = np.std(random_trials)
p_rand = np.mean(random_trials)
# note: std here is between different trials (of multiple actions)
print(f"Avg reward with random actions: {p_rand:.4f} (standard deviation: {random_std:.4f})")

optimal_actions, _ = bauwerk.solve(env)
print(optimal_actions)
p_opt = evaluate_actions(optimal_actions[:eval_length], seed)
print(f"Avg reward (per step) with optimal actions: {p_opt:.4f}")

2022-09-29 18:02:35.404 | WARNING  | bauwerk.envs.distributions:BuildDistBCoreEnv:17 - Deprecation warning: access of building distribution B via gym.make has been deprecated. Use bauwerk.benchmarks.BuildDistB instead.


7.073490235734289


NameError: name 'evaluate_actions' is not defined

### Stable Baselines

In [15]:
EVAL_LEN = 24 * 30

def eval_model(model, env):
    # Obtaining model actions and evaluating them
    model_actions = []
    obs = env.reset()
    for i in range(EVAL_LEN):
        action, _states = model.predict(obs)
        model_actions.append(action)
        obs, _, _, _ = env.step(action)

    p_model = evaluate_actions(model_actions[:EVAL_LEN], env)
    return p_model

def evaluate_actions(actions, seed):
    cum_reward = 0
    eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")
    eval_obs = eval_env.reset()
    for action in actions:
        eval_obs, reward, done, info = eval_env.step(np.array(action, dtype=np.float32))
        
        cum_reward += reward
    
    print('Mean eval reward: {:.4f}'.format(cum_reward / len(actions)))
    return cum_reward / len(actions)

# callback for evaluating callback during training
class EvalCallback(BaseCallback):
    def __init__(self, eval_freq = 24*7, verbose=0):
        super().__init__(verbose)
        self.data = []
        self.eval_freq = eval_freq
        self.eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")

    def _on_training_start(self) -> None:
        """
        This method is called before the first rollout starts.
        """
        self.data.append(eval_model(self.model, self.eval_env))

    def _on_step(self) -> bool:
        if self.num_timesteps % self.eval_freq == 0:
            self.data.append(eval_model(self.model, self.eval_env))

        return True

In [ ]:
from stable_baselines3 import SAC

NUM_TRAIN_STEP = 24 * 365 * 2

model_sac = SAC(
    policy="MultiInputPolicy",
    env="bauwerk/SolarBatteryHouse-v0",
)
sac_callback = EvalCallback()
model_sac.learn(total_timesteps=NUM_TRAIN_STEP,callback=sac_callback)

p_model_sac = eval_model(model_sac, env)

print(f"Avg reward (per step) with model actions: {p_model_sac:.4f}")


### My SAC


In [3]:
def dict_to_array(obs_dict):
    # modify obs
    vals = []
    for _, value in obs_dict.items():
        vals.append(value)
    obs_array = np.concatenate(vals, axis=0, dtype=np.float32)
    
    return obs_array

In [4]:
env = gym.make("bauwerk/SolarBatteryHouse-v0")
env = ObsWrapper(env)

episodes = 2
eval_interval = 24 * 7
eval_steps = 24 * 30
eval_episodes = 0

cfg = parse_cfg()
cfg.sac.obs_dim = 5
cfg.sac.action_dim = 1
cfg.sac.action_range = [
    float(env.action_space.low.min()),
    float(env.action_space.high.max())
]
cfg.eval_frequency = eval_interval
agent = SACAgent(cfg)
replay_buffer = ReplayBuffer(cfg, 5, 1)
step = 0

def evaluate(model):
    eval_steps = 24 * 30
    eval_rewards = 0
    eval_env = gym.make("bauwerk/SolarBatteryHouse-v0")
    eval_env = ObsWrapper(eval_env)
    eval_obs = eval_env.reset()
    actions = []

    for j in range(eval_steps):
        with utils.eval_mode(model):
            # print('eval_obs:', eval_obs)
            action = model.act(eval_obs, sample=False)
            # print('action:', action)
            actions.append(action)
        
        eval_obs, reward, done, _ = eval_env.step(action)
        eval_rewards += reward
    print('action-sum:', sum(actions))
    # print('end of eval')
    # print('/n')

    mean = (eval_rewards / eval_steps)   
    
    return mean

for i in tqdm(range(episodes)):
    ep_reward = 0
    evals = 0
    done = False
    obs = env.reset()
    print('battery size: {:.4f}'.format(env.cfg.battery_size))

    while not done:
        if step < cfg.num_seed_steps:
            action = env.action_space.sample()
        else:
            action = agent.act(obs, sample=True)
        obs_, reward, done, _ = env.step(action)
        
        done = float(done)
        done_no_max = done
            
        replay_buffer.add(obs, action, reward, obs_, done, done_no_max)
        obs = obs_
        ep_reward += reward
 
        if step >= cfg.num_seed_steps:
            agent.update(replay_buffer, step)

        if step % eval_interval == 0:
            eval_episodes += 1
            eval_reward = evaluate(model=agent)
            print('Eval episode {}, mean reward: {}'.format(eval_episodes, eval_reward))
            
        step += 1
            


  0%|          | 0/2 [00:00<?, ?it/s]

battery size: 7.5000
action-sum: [180.698]
Eval episode 1, mean reward: -0.16046422424084691
action-sum: [-595.899]
Eval episode 2, mean reward: -0.15855319001875615
action-sum: [-536.756]
Eval episode 3, mean reward: -0.15855319001875615
action-sum: [-384.101]
Eval episode 4, mean reward: -0.15855319001875615
action-sum: [-387.646]
Eval episode 5, mean reward: -0.15855319001875615
action-sum: [-368.831]
Eval episode 6, mean reward: -0.15855319001875615
action-sum: [-376.976]
Eval episode 7, mean reward: -0.15855319001875615
action-sum: [-355.318]
Eval episode 8, mean reward: -0.15855319001875615
action-sum: [-350.55]
Eval episode 9, mean reward: -0.15855319001875615
action-sum: [-343.727]
Eval episode 10, mean reward: -0.15855319001875615
action-sum: [-375.942]
Eval episode 11, mean reward: -0.15855319001875615
action-sum: [-357.204]
Eval episode 12, mean reward: -0.15855319001875615
action-sum: [-356.302]
Eval episode 13, mean reward: -0.15855319001875615
action-sum: [-320.088]
Eval 

  0%|          | 0/2 [00:22<?, ?it/s]


KeyboardInterrupt: 